# LeetCode 923 — 3Sum With Multiplicity bằng Python

Link đề: https://leetcode.com/problems/3sum-with-multiplicity/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Two Pointers, Sorting, Counting

Skeleton LeetCode:

```python
class Solution:
    def threeSumMulti(self, arr: List[int], target: int) -> int:
        ...
```

Ghi chú: đáp án duy nhất (một số nguyên, lấy **modulo `10^9 + 7`**). Khác bài 3Sum (15), ở đây đếm **bộ chỉ số** `i < j < k` (kể cả các bộ cho cùng bộ giá trị), nên các giá trị trùng nhau đóng góp theo **tổ hợp**: chọn `c` phần tử từ `m` bản sao cho `C(m, c)` cách. Vì `arr[i] ≤ 100` chỉ có **101 giá trị khác nhau**, ta đếm theo giá trị. Oracle: **vét cạn mọi bộ chỉ số** `i < j < k` (`itertools.combinations`), đếm chính xác rồi lấy modulo.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số bộ chỉ số `(i, j, k)` với `i < j < k` và `arr[i] + arr[j] + arr[k] == target`, modulo `10^9 + 7`.
2. Nắm công cụ Python: `collections.Counter`, tổ hợp `C(m, 2) = m(m−1)/2`, `C(m, 3) = m(m−1)(m−2)/6`, `math.comb`, hai con trỏ trên mảng đã sắp xếp, quy hoạch động "chọn đúng `c` phần tử".
3. Đi từ **vét cạn `O(n³)`** → **sắp xếp + hai con trỏ xử lý nhóm trùng** → **đếm theo giá trị `x ≤ y ≤ z` bằng tổ hợp (`O(101²)`)** → **quy hoạch động "ba phần tử có tổng `s`"**.
4. Hiểu mấu chốt: với bộ giá trị `x ≤ y ≤ z` thoả `x + y + z = target`, số bộ chỉ số tương ứng là: `cnt[x]·cnt[y]·cnt[z]` nếu ba giá trị khác nhau; `C(cnt[x], 2)·cnt[z]` nếu `x = y ≠ z`; `cnt[x]·C(cnt[y], 2)` nếu `x ≠ y = z`; `C(cnt[x], 3)` nếu cả ba bằng nhau.
5. Tránh lỗi kinh điển: dùng `cnt³` hay `cnt(cnt−1)(cnt−2)` (không chia `3!`), quên `C(·, 2)` khi hai giá trị bằng nhau, đếm trùng hoán vị (không ép `x ≤ y ≤ z`), hai con trỏ chỉ nhích một bước khi gặp nhóm trùng, và quên `% (10^9 + 7)`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `arr` và số nguyên `target`. Trả về **số bộ** `(i, j, k)` sao cho `i < j < k` và `arr[i] + arr[j] + arr[k] == target`.

Vì đáp án có thể rất lớn, trả về nó **modulo `10^9 + 7`**.

**Ví dụ 1:** `arr = [1,1,2,2,3,3,4,4,5,5]`, `target = 8` → `20`.

- `(1,2,5)` xuất hiện `8` lần; `(1,3,4)` xuất hiện `8` lần; `(2,2,4)` xuất hiện `2` lần; `(2,3,3)` xuất hiện `2` lần.

**Ví dụ 2:** `arr = [1,1,2,2,2,2]`, `target = 5` → `12` (chọn một số `1` trong `2` cách và hai số `2` trong `C(4,2) = 6` cách).

**Ví dụ 3:** `arr = [2,1,3]`, `target = 6` → `1`.

**Ràng buộc:** `3 <= arr.length <= 3000`; `0 <= arr[i] <= 100`; `0 <= target <= 300`.

In [1]:
import itertools
import math
import random
import time
from collections import Counter
from typing import List

MOD = 10 ** 9 + 7
examples = [(([1, 1, 2, 2, 3, 3, 4, 4, 5, 5], 8), 20), (([1, 1, 2, 2, 2, 2], 5), 12), (([2, 1, 3], 6), 1)]
for (a, t), want in examples:
    print(f"arr = {a}, target = {t} -> {want}")

arr = [1, 1, 2, 2, 3, 3, 4, 4, 5, 5], target = 8 -> 20
arr = [1, 1, 2, 2, 2, 2], target = 5 -> 12
arr = [2, 1, 3], target = 6 -> 1


## 3. Kiến thức nền

### 3.1 Đếm bộ chỉ số, không phải bộ giá trị

3Sum (15) đếm bộ **giá trị** khác nhau. Bài này đếm **bộ chỉ số** `i < j < k`: hai bộ chỉ số khác nhau nhưng cho cùng bộ giá trị vẫn tính riêng. Ví dụ `[2,2,2]`, `target = 6` có đúng `1` bộ chỉ số `(0,1,2)`; còn `[1,1,2,2,2,2]` với `(1,2,2)` có `2 · 6 = 12` bộ chỉ số.

### 3.2 Tổ hợp

Số cách chọn `c` phần tử không thứ tự từ `m` phần tử: `C(m, 2) = m(m−1)/2`, `C(m, 3) = m(m−1)(m−2)/6`. Python có `math.comb(m, c)` (và `0` khi `c > m`).

In [2]:
print([math.comb(m, 2) for m in range(6)], [math.comb(m, 3) for m in range(7)], math.comb(2, 3))

[0, 0, 1, 3, 6, 10] [0, 0, 0, 1, 4, 10, 20] 0


### 3.3 Bốn dạng bộ giá trị `x ≤ y ≤ z`

Sắp thứ tự `x ≤ y ≤ z` để mỗi bộ **giá trị** chỉ xét một lần. Số bộ chỉ số:

| dạng | số bộ chỉ số |
|---|---|
| `x < y < z` | `cnt[x] · cnt[y] · cnt[z]` |
| `x = y < z` | `C(cnt[x], 2) · cnt[z]` |
| `x < y = z` | `cnt[x] · C(cnt[y], 2)` |
| `x = y = z` | `C(cnt[x], 3)` |

Lý do: ba vị trí phải thuộc các nhóm giá trị tương ứng; với giá trị lặp lại ta chọn **tập** vị trí (không thứ tự), và thứ tự `i < j < k` được xác định duy nhất từ tập đó.

### 3.4 Modulo

Tổng cuối cùng có thể lớn (`C(3000, 3) ≈ 4.5·10^9 > 10^9 + 7`); lấy `% MOD` ở cuối (Python không tràn) hoặc sau mỗi phép cộng.

In [3]:
cnt = Counter([1, 1, 2, 2, 3, 3, 4, 4, 5, 5])
x, y, z = 2, 2, 4
print("(2,2,4): C(cnt[2],2) * cnt[4] =", math.comb(cnt[2], 2) * cnt[4], "| (1,2,5):", cnt[1] * cnt[2] * cnt[5], "| (2,3,3):", cnt[2] * math.comb(cnt[3], 2))

(2,2,4): C(cnt[2],2) * cnt[4] = 2 | (1,2,5): 8 | (2,3,3): 2


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1: `arr = [1,1,2,2,3,3,4,4,5,5]`, `target = 8`; mỗi giá trị `1..5` xuất hiện `2` lần.

Các bộ giá trị `x ≤ y ≤ z` có tổng `8`:

- `(1,2,5)`: ba giá trị khác nhau ⇒ `2·2·2 = 8`.
- `(1,3,4)`: `2·2·2 = 8`.
- `(2,2,4)`: `x = y` ⇒ `C(2,2)·2 = 1·2 = 2`.
- `(2,3,3)`: `y = z` ⇒ `2·C(2,2) = 2`.
- `(1,1,6)`, `(1,7,0)`...: không tồn tại hoặc ngoài khoảng.

Tổng `8 + 8 + 2 + 2 = 20` ✓. Nhờ chỉ có `101` giá trị, ta chỉ duyệt `~5000` cặp `(x, y)` thay vì `n³ ≈ 4.5·10^9` bộ.

In [4]:
cnt = Counter([1, 1, 2, 2, 3, 3, 4, 4, 5, 5])
target = 8
total = 0
for x in range(0, 101):
    for y in range(x, 101):
        z = target - x - y
        if z < y or z > 100:
            continue
        a, b, c = cnt[x], cnt[y], cnt[z]
        if x == y == z:
            ways = math.comb(a, 3)
        elif x == y:
            ways = math.comb(a, 2) * c
        elif y == z:
            ways = a * math.comb(b, 2)
        else:
            ways = a * b * c
        if ways:
            print(f"(x,y,z) = ({x},{y},{z}): {ways}")
        total += ways
print("tổng =", total)

(x,y,z) = (1,2,5): 8
(x,y,z) = (1,3,4): 8
(x,y,z) = (2,2,4): 2
(x,y,z) = (2,3,3): 2
tổng = 20


## 5. Cách 1 — Brute Force: mọi bộ chỉ số

**Ý tưởng:** ba vòng lặp `i < j < k`, đếm bộ có tổng bằng `target`. Rút gọn một chút: với `i, j` cố định, số `k` cần tìm là `target − arr[i] − arr[j]`.

In [5]:
def three_sum_brute(arr, target):
    n = len(arr)
    cnt = 0
    for i in range(n):
        for j in range(i + 1, n):
            need = target - arr[i] - arr[j]
            for k in range(j + 1, n):
                if arr[k] == need:
                    cnt += 1
    return cnt % MOD


for (a, t), want in examples:
    assert three_sum_brute(a, t) == want
print([three_sum_brute(a, t) for (a, t), _ in examples])

[20, 12, 1]


In [6]:
# Code nộp LeetCode cho Cách 1 (O(n^3): 4.5·10^9 phép ở n = 3000, quá chậm)
class SolutionBrute:
    def threeSumMulti(self, arr: List[int], target: int) -> int:
        n, cnt = len(arr), 0
        for i in range(n):
            for j in range(i + 1, n):
                need = target - arr[i] - arr[j]
                for k in range(j + 1, n):
                    if arr[k] == need:
                        cnt += 1
        return cnt % (10 ** 9 + 7)

**Độ phức tạp:** `O(n³)` thời gian, `O(1)` bộ nhớ.

**Nút thắt:** ta đếm từng bộ chỉ số một; nhiều bộ chỉ số có cùng bộ giá trị và đếm được **theo nhóm** bằng tổ hợp.

## 6. Ý tưởng thuật toán / bước đột phá

Giá trị nhỏ (`≤ 100`) ⇒ chỉ `101` loại. Đếm `cnt[v]`, rồi duyệt các bộ giá trị `x ≤ y ≤ z` với `x + y + z = target` và cộng theo bảng ở mục 3.3 — `O(101²)`. Hoặc sắp xếp rồi hai con trỏ gom nhóm trùng — `O(n²)`; hoặc quy hoạch động "chọn đúng 3 phần tử có tổng `s`" — `O(n · target)`.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** sắp xếp + hai con trỏ (nhóm trùng), **Cách 3** đếm theo giá trị `x ≤ y ≤ z`, **Cách 4** quy hoạch động chọn 3 phần tử.

### Cách 2 — Sắp xếp + hai con trỏ xử lý nhóm trùng

**Giải thích từng đoạn:**

1. `arr.sort()`. Với mỗi `i`, đặt `T = target − arr[i]`, `j = i + 1`, `k = n − 1`.
2. Khi `j < k`: `s = arr[j] + arr[k]`. Nếu `s < T` thì `j += 1`; nếu `s > T` thì `k -= 1`.
3. Nếu `s == T`: nếu `arr[j] != arr[k]` thì đếm độ dài nhóm trùng bên trái `L` (từ `j`) và bên phải `R` (từ `k`), cộng `L·R`, nhảy `j += L`, `k -= R`. Nếu `arr[j] == arr[k]` thì mọi phần tử trong `[j, k]` bằng nhau: cộng `C(k − j + 1, 2)` rồi dừng.

In [7]:
def three_sum_two_pointers(arr, target):
    a = sorted(arr)
    n, ans = len(a), 0
    for i in range(n - 2):
        T = target - a[i]
        j, k = i + 1, n - 1
        while j < k:
            s = a[j] + a[k]
            if s < T:
                j += 1
            elif s > T:
                k -= 1
            elif a[j] != a[k]:
                L = R = 1
                while j + L < k and a[j + L] == a[j]:
                    L += 1                               # độ dài nhóm trùng bên trái
                while k - R > j and a[k - R] == a[k]:
                    R += 1                               # độ dài nhóm trùng bên phải
                ans += L * R
                j += L
                k -= R
            else:                                        # a[j] == a[k]: cả đoạn [j, k] cùng giá trị
                m = k - j + 1
                ans += m * (m - 1) // 2
                break
    return ans % MOD


for (a, t), want in examples:
    assert three_sum_two_pointers(a, t) == want
print([three_sum_two_pointers(a, t) for (a, t), _ in examples])

[20, 12, 1]


In [8]:
# Code nộp LeetCode cho Cách 2
class SolutionTwoPointers:
    def threeSumMulti(self, arr: List[int], target: int) -> int:
        arr.sort()
        n, ans = len(arr), 0
        for i in range(n - 2):
            T = target - arr[i]
            j, k = i + 1, n - 1
            while j < k:
                s = arr[j] + arr[k]
                if s < T:
                    j += 1
                elif s > T:
                    k -= 1
                elif arr[j] != arr[k]:
                    L = R = 1
                    while j + L < k and arr[j + L] == arr[j]:
                        L += 1
                    while k - R > j and arr[k - R] == arr[k]:
                        R += 1
                    ans += L * R
                    j += L
                    k -= R
                else:
                    m = k - j + 1
                    ans += m * (m - 1) // 2
                    break
        return ans % (10 ** 9 + 7)

**Độ phức tạp:** `O(n²)` thời gian (mỗi `i` hai con trỏ chạy `O(n)`), `O(1)` bộ nhớ ngoài sắp xếp. Với `n = 3000`: `~4.5·10^6` bước.

### Cách 3 — Đếm theo giá trị `x ≤ y ≤ z` bằng tổ hợp

**Ý tưởng:** `cnt` = số lần xuất hiện mỗi giá trị `0..100`; với mỗi cặp `x ≤ y`, đặt `z = target − x − y`; nếu `y ≤ z ≤ 100` cộng số bộ chỉ số theo bảng 3.3. Chi phí chỉ `101²/2 ≈ 5000`.

In [9]:
def three_sum_counting(arr, target):
    cnt = [0] * 101
    for v in arr:
        cnt[v] += 1
    ans = 0
    for x in range(101):
        for y in range(x, 101):
            z = target - x - y
            if z < y or z > 100:                         # ép x <= y <= z: mỗi bộ GIÁ TRỊ chỉ xét một lần
                continue
            a, b, c = cnt[x], cnt[y], cnt[z]
            if x == y == z:
                ans += a * (a - 1) * (a - 2) // 6        # C(a, 3)
            elif x == y:
                ans += a * (a - 1) // 2 * c              # C(a, 2) * cnt[z]
            elif y == z:
                ans += a * (b * (b - 1) // 2)            # cnt[x] * C(b, 2)
            else:
                ans += a * b * c
    return ans % MOD


for (a, t), want in examples:
    assert three_sum_counting(a, t) == want
print([three_sum_counting(a, t) for (a, t), _ in examples])

[20, 12, 1]


In [10]:
# Code nộp LeetCode cho Cách 3
class SolutionCounting:
    def threeSumMulti(self, arr: List[int], target: int) -> int:
        cnt = [0] * 101
        for v in arr:
            cnt[v] += 1
        ans = 0
        for x in range(101):
            for y in range(x, 101):
                z = target - x - y
                if z < y or z > 100:
                    continue
                a, b, c = cnt[x], cnt[y], cnt[z]
                if x == y == z:
                    ans += a * (a - 1) * (a - 2) // 6
                elif x == y:
                    ans += a * (a - 1) // 2 * c
                elif y == z:
                    ans += a * (b * (b - 1) // 2)
                else:
                    ans += a * b * c
        return ans % (10 ** 9 + 7)

**Độ phức tạp:** `O(n + 101²)` thời gian, `O(101)` bộ nhớ. Nhanh nhất khi giá trị nhỏ (`≤ 100`).

### Cách 4 — Quy hoạch động chọn đúng 3 phần tử

**Ý tưởng:** `ways[c][s]` = số cách chọn **đúng `c` phần tử** (theo chỉ số, thứ tự tăng) trong các phần tử đã xét có tổng `s`. Với mỗi phần tử `x`, cập nhật `c` từ `3` xuống `1` (để mỗi phần tử dùng tối đa một lần): `ways[c][s] += ways[c−1][s − x]`. Đáp án `ways[3][target]`.

In [11]:
def three_sum_dp(arr, target):
    ways = [[0] * (target + 1) for _ in range(4)]
    ways[0][0] = 1                                       # chọn 0 phần tử, tổng 0: một cách
    for x in arr:
        for c in range(3, 0, -1):                        # c giảm dần: mỗi phần tử dùng tối đa một lần
            prev, cur = ways[c - 1], ways[c]
            for s in range(target, x - 1, -1):
                cur[s] += prev[s - x]
    return ways[3][target] % MOD


for (a, t), want in examples:
    assert three_sum_dp(a, t) == want
print([three_sum_dp(a, t) for (a, t), _ in examples])

[20, 12, 1]


In [12]:
# Code nộp LeetCode cho Cách 4
class SolutionDP:
    def threeSumMulti(self, arr: List[int], target: int) -> int:
        ways = [[0] * (target + 1) for _ in range(4)]
        ways[0][0] = 1
        for x in arr:
            for c in range(3, 0, -1):
                prev, cur = ways[c - 1], ways[c]
                for s in range(target, x - 1, -1):
                    cur[s] += prev[s - x]
        return ways[3][target] % (10 ** 9 + 7)

**Độ phức tạp:** `O(n · 3 · target)` = `3000 · 3 · 300 = 2.7·10^6` thời gian, `O(3 · target)` bộ nhớ. Tổng quát cho mọi `k` phần tử (`k-Sum` đếm bộ chỉ số).

## 8. TRACE từng bước

**Trace Cách 3** (đã in ở mục 4) cho ví dụ 1. **Trace Cách 2** với `arr = [1,1,2,2,2,2]`, `target = 5` (đã sắp xếp):

| `i` | `a[i]` | `T = 5 − a[i]` | `(j, k)` | `a[j] + a[k]` | quyết định | cộng |
|---|---|---|---|---|---|---|
| 0 | 1 | 4 | `(1, 5)` | `1 + 2 = 3 < 4` | `j = 2` | — |
| 0 | 1 | 4 | `(2, 5)` | `2 + 2 = 4` | `a[j] == a[k]`: `m = 4`, `C(4,2) = 6` | 6, dừng |
| 1 | 1 | 4 | `(2, 5)` | `4` | `a[j] == a[k]`: `m = 4` | 6, dừng |
| 2.. | 2 | 3 | không có cặp tổng `3` | | | — |

Tổng `12` ✓. Hàm trace:

In [13]:
def trace_two_pointers(arr, target):
    a = sorted(arr)
    n, ans = len(a), 0
    print(f"a = {a}, target = {target}")
    for i in range(n - 2):
        T = target - a[i]
        j, k = i + 1, n - 1
        print(f"i={i} (a[i]={a[i]}), T={T}")
        while j < k:
            s = a[j] + a[k]
            if s < T:
                print(f"   (j={j},k={k}) {a[j]}+{a[k]}={s} < T -> j += 1")
                j += 1
            elif s > T:
                print(f"   (j={j},k={k}) {a[j]}+{a[k]}={s} > T -> k -= 1")
                k -= 1
            elif a[j] != a[k]:
                L = R = 1
                while j + L < k and a[j + L] == a[j]:
                    L += 1
                while k - R > j and a[k - R] == a[k]:
                    R += 1
                print(f"   (j={j},k={k}) {a[j]}+{a[k]}={s} == T, giá trị khác nhau: nhóm trái {L} x nhóm phải {R} = {L * R}")
                ans += L * R
                j += L
                k -= R
            else:
                m = k - j + 1
                print(f"   (j={j},k={k}) {a[j]}+{a[k]}={s} == T, cả đoạn [{j},{k}] cùng giá trị: C({m},2) = {m * (m - 1) // 2}")
                ans += m * (m - 1) // 2
                break
    print("=> tổng =", ans)
    return ans


assert trace_two_pointers([1, 1, 2, 2, 2, 2], 5) == 12

a = [1, 1, 2, 2, 2, 2], target = 5
i=0 (a[i]=1), T=4
   (j=1,k=5) 1+2=3 < T -> j += 1
   (j=2,k=5) 2+2=4 == T, cả đoạn [2,5] cùng giá trị: C(4,2) = 6
i=1 (a[i]=1), T=4
   (j=2,k=5) 2+2=4 == T, cả đoạn [2,5] cùng giá trị: C(4,2) = 6
i=2 (a[i]=2), T=3
   (j=3,k=5) 2+2=4 > T -> k -= 1
   (j=3,k=4) 2+2=4 > T -> k -= 1
i=3 (a[i]=2), T=3
   (j=4,k=5) 2+2=4 > T -> k -= 1
=> tổng = 12


## 9. Code nộp LeetCode

Chọn **Cách 3** (đếm theo giá trị, `O(n + 101²)`). Giữ nguyên `class Solution`, `threeSumMulti(self, arr, target)`:

In [14]:
class Solution:
    def threeSumMulti(self, arr: List[int], target: int) -> int:
        MOD = 10 ** 9 + 7
        cnt = [0] * 101                                   # cnt[v] = số lần giá trị v xuất hiện
        for v in arr:
            cnt[v] += 1
        ans = 0
        for x in range(101):
            for y in range(x, 101):                       # ép x <= y <= z để mỗi bộ giá trị chỉ xét một lần
                z = target - x - y
                if z < y or z > 100:
                    continue
                a, b, c = cnt[x], cnt[y], cnt[z]
                if x == y == z:
                    ans += a * (a - 1) * (a - 2) // 6     # C(a, 3)
                elif x == y:
                    ans += a * (a - 1) // 2 * c           # C(a, 2) * cnt[z]
                elif y == z:
                    ans += a * (b * (b - 1) // 2)         # cnt[x] * C(b, 2)
                else:
                    ans += a * b * c
        return ans % MOD


print([Solution().threeSumMulti(a, t) for (a, t), _ in examples])      # [20, 12, 1]
assert all(Solution().threeSumMulti(list(a), t) == want for (a, t), want in examples)

[20, 12, 1]


## 10. Vì sao đúng

**(a) Phân hoạch theo bộ giá trị.** Mỗi bộ chỉ số `i < j < k` ứng với bộ giá trị `(arr[i], arr[j], arr[k])`; sắp xếp tăng dần bộ này được `(x, y, z)` với `x ≤ y ≤ z`. Hai bộ chỉ số cho cùng một **đa tập** giá trị `{x, y, z}` thuộc cùng một lớp. Các lớp rời nhau theo `(x ≤ y ≤ z)`, nên tổng bằng tổng số bộ chỉ số trong từng lớp.

**(b) Số bộ chỉ số trong một lớp.** Một bộ chỉ số `i < j < k` thuộc lớp `(x, y, z)` khi tập ba chỉ số `{i, j, k}` có các giá trị là đa tập `{x, y, z}`. Số **tập** ba chỉ số như vậy:
- `x < y < z`: chọn một chỉ số mỗi nhóm: `cnt[x]·cnt[y]·cnt[z]`.
- `x = y < z`: chọn **hai** chỉ số từ nhóm `x` (không thứ tự) và một từ nhóm `z`: `C(cnt[x], 2)·cnt[z]`.
- `x < y = z`: `cnt[x]·C(cnt[y], 2)`.
- `x = y = z`: `C(cnt[x], 3)`.
Mỗi **tập** ba chỉ số cho đúng **một** bộ `i < j < k` (sắp tăng dần), nên số bộ chỉ số bằng số tập.

**(c) Ép `x ≤ y ≤ z`.** Vòng lặp `x`, `y ≥ x`, `z = target − x − y ≥ y` duyệt đúng một lần mỗi đa tập; điều kiện `z ≤ 100` (và `z ≥ y ≥ 0`) loại giá trị ngoài miền.

**(d) Cách 2.** Với `i` cố định và `a` đã sắp, các cặp `(j, k)`, `i < j < k` có `a[j] + a[k] = T`: nếu `a[j] ≠ a[k]` thì mọi cách chọn một chỉ số từ nhóm trùng của `a[j]` (bên phải `i`... trong `[j, k)`) và một từ nhóm của `a[k]` đều hợp lệ (`L·R`); nếu `a[j] = a[k]` thì mọi phần tử giữa bằng nhau và số cặp là `C(m, 2)`. Hai nhóm không giao nhau khi `a[j] ≠ a[k]` (vì mảng sắp tăng), và nhảy `j += L`, `k -= R` giữ nguyên tính đúng cho các cặp còn lại.

**(e) Cách 4.** `ways[c][s]` sau khi xét `m` phần tử đầu = số tập `c` chỉ số trong `m` phần tử đầu có tổng `s` (quy nạp: phần tử `x` hoặc không dùng, hoặc là phần tử cuối của tập, đóng góp `ways[c−1][s−x]` trước khi xét `x`). Cập nhật `c` giảm dần dùng giá trị **cũ** của `ways[c−1]`, nên mỗi phần tử dùng tối đa một lần.

**(f) Modulo.** Đáp án thật là tổng các số tự nhiên; Python tính chính xác rồi `% MOD` một lần.

Ta kiểm chứng bằng máy: mọi cách khớp oracle **mọi bộ chỉ số** `i < j < k` (`itertools.combinations`) trên mọi mảng nhỏ (mục 12D) và ngẫu nhiên (12E); và khớp nhau ở `n = 3000` (12F).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mọi bộ chỉ số | `O(n³)` | `O(1)` | oracle |
| 2. Sắp xếp + hai con trỏ | `O(n²)` | `O(1)` | không dùng giá trị `≤ 100` |
| 3. Đếm theo giá trị | `O(n + 101²)` | `O(101)` | **đáp án chính** |
| 4. DP chọn 3 phần tử | `O(3·n·target)` | `O(3·target)` | tổng quát `k-Sum` |

Đo thực tế (`n = 3000`; cách 1 chỉ `n = 150`):

In [15]:
def bench(fn, args, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(*args)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(923)
big = [rng.randint(0, 100) for _ in range(3000)]
for name, fn in [("two_pointers", three_sum_two_pointers), ("counting", three_sum_counting), ("dp", three_sum_dp)]:
    print(f"n = 3000 {name:13s} {bench(fn, (big, 150)) * 1e3:9.2f} ms")
print(f"cách 1 với n = 150: {bench(three_sum_brute, (big[:150], 150), 1) * 1e3:8.1f} ms (tăng theo n^3)")

n = 3000 two_pointers     308.64 ms
n = 3000 counting           0.40 ms
n = 3000 dp                39.53 ms
cách 1 với n = 150:     18.1 ms (tăng theo n^3)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi mảng nhỏ và mọi `target` với oracle `combinations`, (E) ngẫu nhiên có seed + tính chất, (F) giới hạn (`n = 3000`).

### A. Ví dụ của đề

In [16]:
def oracle(arr, target):
    """Đúng định nghĩa: đếm các bộ chỉ số (i, j, k), i < j < k, tổng = target (itertools.combinations đảm bảo i < j < k); modulo."""
    return sum(1 for a, b, c in itertools.combinations(arr, 3) if a + b + c == target) % MOD


IMPLS = {
    "brute": three_sum_brute,
    "two_pointers": three_sum_two_pointers,
    "counting": three_sum_counting,
    "dp": three_sum_dp,
    "Solution": lambda a, t: Solution().threeSumMulti(list(a), t),
    "SolutionBrute": lambda a, t: SolutionBrute().threeSumMulti(list(a), t),
    "SolutionTwoPointers": lambda a, t: SolutionTwoPointers().threeSumMulti(list(a), t),
    "SolutionCounting": lambda a, t: SolutionCounting().threeSumMulti(list(a), t),
    "SolutionDP": lambda a, t: SolutionDP().threeSumMulti(list(a), t),
}


def check(arr, target, want=None, names=None):
    if want is None:
        want = oracle(arr, target)
    for name in (names or IMPLS):
        got = IMPLS[name](list(arr), target)
        assert got == want, (name, arr, target, got, want)


for (a, t), want in examples:
    assert oracle(a, t) == want
    check(a, t, want)
print("ví dụ đề: oracle combinations và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle combinations và 9 cài đặt đúng


### B. Edge case

- `n = 3`: một bộ duy nhất (đúng hoặc không).
- Mọi phần tử bằng nhau và `3v = target`: `C(n, 3)`.
- `target = 0` với toàn `0`.
- `target = 300` với toàn `100`.
- Không có bộ nào (đáp án `0`).
- Giá trị `0` lẫn giá trị khác.
- Hai giá trị: `x = y ≠ z` và `x ≠ y = z`.
- Mảng chưa sắp xếp, thứ tự bất kỳ (kết quả không phụ thuộc thứ tự).

In [17]:
cases = [
    ([1, 2, 3], 6, 1), ([1, 2, 3], 7, 0), ([0, 0, 0], 0, 1), ([0] * 10, 0, math.comb(10, 3)), ([100] * 8, 300, math.comb(8, 3)),
    ([5, 5, 5, 5], 15, 4), ([1, 1, 2], 4, 1), ([1, 2, 2], 5, 1), ([1, 1, 1, 2, 2], 5, 3), ([2, 2, 1, 1, 1], 5, 3),
    ([3, 3, 3, 3, 3, 3], 9, 20), ([0, 1, 2, 3, 4, 5], 6, None), ([100, 100, 100, 0], 200, 3), ([7, 7, 7], 22, 0),
]
for a, t, want in cases:
    check(a, t, want)
rng = random.Random(1)
a = [rng.randint(0, 10) for _ in range(12)]
b = a[:]
rng.shuffle(b)
assert Solution().threeSumMulti(a, 15) == Solution().threeSumMulti(b, 15) == oracle(a, 15)
print(len(cases), "edge case +1: đúng, kết quả không phụ thuộc thứ tự")

14 edge case +1: đúng, kết quả không phụ thuộc thứ tự


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [18]:
def counting_variant(arr, target, f_same3, f_xy, f_yz, f_diff, ordered=True):
    cnt = [0] * 101
    for v in arr:
        cnt[v] += 1
    ans = 0
    for x in range(101):
        for y in (range(x, 101) if ordered else range(101)):
            z = target - x - y
            if (z < y if ordered else not (0 <= z <= 100)) or z > 100:
                continue
            a, b, c = cnt[x], cnt[y], cnt[z]
            if x == y == z:
                ans += f_same3(a)
            elif x == y:
                ans += f_xy(a, c)
            elif y == z:
                ans += f_yz(a, b)
            else:
                ans += f_diff(a, b, c)
    return ans % MOD


C2 = lambda m: m * (m - 1) // 2
C3 = lambda m: m * (m - 1) * (m - 2) // 6


def bug_cubed(arr, target):                            # cnt^3 thay vì C(cnt, 3)
    return counting_variant(arr, target, lambda a: a ** 3, lambda a, c: C2(a) * c, lambda a, b: a * C2(b), lambda a, b, c: a * b * c)


def bug_no_divide(arr, target):                        # cnt(cnt-1)(cnt-2) không chia 3!
    return counting_variant(arr, target, lambda a: a * (a - 1) * (a - 2), lambda a, c: C2(a) * c, lambda a, b: a * C2(b), lambda a, b, c: a * b * c)


def bug_pair_ordered(arr, target):                     # cnt(cnt-1) thay vì C(cnt, 2) (đếm có thứ tự)
    return counting_variant(arr, target, C3, lambda a, c: a * (a - 1) * c, lambda a, b: a * b * (b - 1), lambda a, b, c: a * b * c)


def bug_no_pair_comb(arr, target):                     # x == y: dùng cnt[x] * cnt[z] (bỏ tổ hợp)
    return counting_variant(arr, target, C3, lambda a, c: a * c, lambda a, b: a * b, lambda a, b, c: a * b * c)


def bug_unordered_double(arr, target):                 # không ép x <= y <= z: đếm lặp theo hoán vị
    cnt = [0] * 101
    for v in arr:
        cnt[v] += 1
    ans = 0
    for x in range(101):
        for y in range(101):
            z = target - x - y
            if not 0 <= z <= 100:
                continue
            a, b, c = cnt[x], cnt[y], cnt[z]
            if x == y == z:
                ans += C3(a)
            elif x == y:
                ans += C2(a) * c
            elif y == z:
                ans += a * C2(b)
            else:
                ans += a * b * c
    return ans % MOD


def bug_two_pointers_step_one(arr, target):            # gặp tổng bằng thì chỉ nhích một bước (bỏ qua nhóm trùng)
    a = sorted(arr)
    n, ans = len(a), 0
    for i in range(n - 2):
        T = target - a[i]
        j, k = i + 1, n - 1
        while j < k:
            s = a[j] + a[k]
            if s < T:
                j += 1
            elif s > T:
                k -= 1
            else:
                ans += 1                               # SAI: mỗi cặp (j, k) chỉ đếm một lần
                j += 1
                k -= 1
    return ans % MOD


def bug_no_mod(arr, target):                           # quên modulo
    cnt = [0] * 101
    for v in arr:
        cnt[v] += 1
    return sum(C3(cnt[x]) for x in range(101) if 3 * x == target)


BUGS = [bug_cubed, bug_no_divide, bug_pair_ordered, bug_no_pair_comb, bug_unordered_double, bug_two_pointers_step_one]
rng = random.Random(3)
tests = [([rng.randint(0, 4) for _ in range(rng.randint(3, 9))], rng.randint(0, 12)) for _ in range(1000)]
for bug in BUGS:
    bad = next(((a, t) for a, t in tests if bug(a, t) != oracle(a, t)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:26s} sai trên arr = {bad[0]}, target = {bad[1]}: ra {bug(*bad)}, đúng {oracle(*bad)}")
big_eq = [50] * 3000
print("bug_no_mod (3000 số 50, target 150): ra", bug_no_mod(big_eq, 150), "| đúng:", Solution().threeSumMulti(big_eq, 150), "| C(3000,3) =", math.comb(3000, 3))
assert bug_no_mod(big_eq, 150) != Solution().threeSumMulti(big_eq, 150)

bug_cubed                  sai trên arr = [3, 2, 3, 4, 3, 4, 2, 4], target = 9: ra 45, đúng 19
bug_no_divide              sai trên arr = [3, 2, 3, 4, 3, 4, 2, 4], target = 9: ra 24, đúng 19
bug_pair_ordered           sai trên arr = [4, 4, 1, 2], target = 9: ra 2, đúng 1
bug_no_pair_comb           sai trên arr = [4, 4, 1, 2], target = 9: ra 2, đúng 1
bug_unordered_double       sai trên arr = [4, 4, 1, 2], target = 9: ra 6, đúng 1
bug_two_pointers_step_one  sai trên arr = [3, 2, 3, 4, 3, 4, 2, 4], target = 9: ra 7, đúng 19
bug_no_mod (3000 số 50, target 150): ra 4495501000 | đúng: 495500972 | C(3000,3) = 4495501000


### D. Vét cạn mọi mảng nhỏ

Mọi mảng độ dài `3..6` với giá trị `0..3` (`64 + 256 + 1 024 + 4 096 = 5 440` mảng), với **mọi** `target` từ `0` tới `10` (`11` giá trị) — `59 840` bộ. Oracle: `combinations`.

In [19]:
t0 = time.perf_counter()
cnt = 0
fast = ("brute", "two_pointers", "counting", "dp", "Solution")
for n in range(3, 7):
    for tup in itertools.product(range(4), repeat=n):
        a = list(tup)
        for target in range(11):
            want = oracle(a, target)
            for name in fast:
                assert IMPLS[name](a, target) == want, (name, a, target)
            cnt += 1
print(cnt, "bộ (mảng, target): 5 cài đặt khớp oracle combinations;", round(time.perf_counter() - t0, 1), "s")
assert cnt == 5440 * 11

59840 bộ (mảng, target): 5 cài đặt khớp oracle combinations; 29.5 s


### E. Ngẫu nhiên có seed và tính chất

Mảng ngẫu nhiên tới `40` phần tử (so oracle) và tới `300` phần tử (so Cách 1), giá trị `0..100` hoặc dồn vào ít giá trị; `target` ngẫu nhiên `0..300`. Tính chất: (1) xáo trộn mảng không đổi đáp án; (2) tổng đáp án theo mọi `target` bằng `C(n, 3)` (mỗi bộ chỉ số có đúng một tổng); (3) thêm một phần tử `v` làm đáp án tăng đúng số cặp `(i, j)` trước đó có tổng `target − v`.

In [20]:
rng = random.Random(20240923)
for _ in range(1500):
    n = rng.randint(3, 40)
    hi = rng.choice([3, 10, 100])
    a = [rng.randint(0, hi) for _ in range(n)]
    t = rng.randint(0, min(300, 3 * hi + 2))
    want = oracle(a, t)
    for name in ("two_pointers", "counting", "dp", "Solution", "SolutionTwoPointers", "SolutionCounting", "SolutionDP"):
        assert IMPLS[name](a, t) == want, (name, a, t)
    b = a[:]
    rng.shuffle(b)
    assert Solution().threeSumMulti(b, t) == want                                          # (1)
for _ in range(60):
    n = rng.randint(3, 25)
    a = [rng.randint(0, 6) for _ in range(n)]
    assert sum(Solution().threeSumMulti(a, t) for t in range(0, 19)) % MOD == math.comb(n, 3) % MOD      # (2)
    v = rng.randint(0, 6)
    t = rng.randint(0, 18)
    pairs = sum(1 for x, y in itertools.combinations(a, 2) if x + y == t - v)
    assert Solution().threeSumMulti(a + [v], t) == (Solution().threeSumMulti(a, t) + pairs) % MOD     # (3)
for _ in range(60):
    a = [rng.randint(0, 100) for _ in range(rng.randint(100, 300))]
    t = rng.randint(100, 200)
    assert Solution().threeSumMulti(a, t) == three_sum_brute(a, t) == three_sum_two_pointers(a, t) == three_sum_dp(a, t)
print("1500 mảng ngẫu nhiên khớp oracle; 3 tính chất (xáo trộn, Σ target = C(n,3), thêm phần tử) và 60 mảng tới 300 phần tử đúng")

1500 mảng ngẫu nhiên khớp oracle; 3 tính chất (xáo trộn, Σ target = C(n,3), thêm phần tử) và 60 mảng tới 300 phần tử đúng


### F. Giới hạn: `n = 3000`

Các cách `O(n²)` hoặc nhanh hơn khớp nhau trên nhiều dạng dữ liệu; công thức đóng cho dạng đặc biệt; kiểm tra việc lấy modulo thực sự cần thiết.

In [21]:
rng = random.Random(500)
n = 3000
fast = ["two_pointers", "counting", "dp", "Solution", "SolutionTwoPointers", "SolutionCounting", "SolutionDP"]
datasets = {
    "ngẫu nhiên 0..100": [rng.randint(0, 100) for _ in range(n)],
    "ngẫu nhiên 0..3": [rng.randint(0, 3) for _ in range(n)],
    "toàn bằng nhau 50": [50] * n,
    "hai giá trị 0/100": [rng.choice([0, 100]) for _ in range(n)],
    "tăng dần cyclic": [i % 101 for i in range(n)],
}
for name, a in datasets.items():
    for t in (0, 1, 150, 200, 299, 300):
        res = {k: IMPLS[k](a, t) for k in fast}
        assert len(set(res.values())) == 1, (name, t, res)
    print(f"{name:20s}: target 150 -> {IMPLS['counting'](a, 150)}")
assert Solution().threeSumMulti([50] * n, 150) == math.comb(n, 3) % MOD != math.comb(n, 3)
assert Solution().threeSumMulti([0] * n, 0) == math.comb(n, 3) % MOD
assert Solution().threeSumMulti([100] * n, 300) == math.comb(n, 3) % MOD
assert Solution().threeSumMulti([50] * n, 149) == 0
print("công thức đóng khớp; C(3000,3) =", math.comb(n, 3), "> MOD nên modulo là cần thiết")

ngẫu nhiên 0..100   : target 150 -> 33718746


ngẫu nhiên 0..3     : target 150 -> 0


toàn bằng nhau 50   : target 150 -> 495500972


hai giá trị 0/100   : target 150 -> 0


tăng dần cyclic     : target 150 -> 33523285
công thức đóng khớp; C(3000,3) = 4495501000 > MOD nên modulo là cần thiết


## 13. Các lỗi người mới hay gặp

### Lỗi 1: `cnt³` thay vì `C(cnt, 3)`

In [22]:
print("[5,5,5,5], target 15:", bug_cubed([5, 5, 5, 5], 15), "| đúng:", Solution().threeSumMulti([5, 5, 5, 5], 15))
assert bug_cubed([5, 5, 5, 5], 15) == 64 and Solution().threeSumMulti([5, 5, 5, 5], 15) == 4

[5,5,5,5], target 15: 64 | đúng: 4


**Vì sao sai:** `cnt³` đếm bộ có thứ tự, cho phép lặp chỉ số. **Đúng:** `C(cnt, 3)` = chọn **tập** ba chỉ số khác nhau.

### Lỗi 2: quên chia `3!`

In [23]:
print("[5,5,5,5], target 15:", bug_no_divide([5, 5, 5, 5], 15), "| đúng:", Solution().threeSumMulti([5, 5, 5, 5], 15))
assert bug_no_divide([5, 5, 5, 5], 15) == 24

[5,5,5,5], target 15: 24 | đúng: 4


**Vì sao sai:** `cnt(cnt−1)(cnt−2)` đếm bộ chỉ số có thứ tự (`3! = 6` lần mỗi tập). **Đúng:** chia `6`.

### Lỗi 3: `C(cnt, 2)` viết thành `cnt(cnt−1)` hoặc bỏ tổ hợp

In [24]:
bad = next((a, t) for a, t in tests if bug_pair_ordered(a, t) != oracle(a, t))
print(bad, "->", bug_pair_ordered(*bad), "| đúng:", oracle(*bad))
bad = next((a, t) for a, t in tests if bug_no_pair_comb(a, t) != oracle(a, t))
print(bad, "(bỏ tổ hợp) ->", bug_no_pair_comb(*bad), "| đúng:", oracle(*bad))

([4, 4, 1, 2], 9) -> 2 | đúng: 1
([4, 4, 1, 2], 9) (bỏ tổ hợp) -> 2 | đúng: 1


**Vì sao sai:** hai phần tử cùng giá trị phải chọn thành **tập** (không thứ tự): `C(cnt, 2) = cnt(cnt−1)/2`. **Đúng:** `a * (a - 1) // 2`.

### Lỗi 4: không ép `x ≤ y ≤ z` — đếm lặp theo hoán vị

In [25]:
print("[1,2,3], target 6:", bug_unordered_double([1, 2, 3], 6), "| đúng:", Solution().threeSumMulti([1, 2, 3], 6))
assert bug_unordered_double([1, 2, 3], 6) == 6 and Solution().threeSumMulti([1, 2, 3], 6) == 1     # 3! hoán vị của (1,2,3)

[1,2,3], target 6: 6 | đúng: 1


**Vì sao sai:** `(1,2,3)`, `(1,3,2)`, `(2,1,3)`, … là cùng một đa tập; không ép thứ tự thì mỗi lớp bị đếm tới `3!` lần (hoặc `3` lần khi hai giá trị bằng nhau). **Đúng:** `y ≥ x` và `z ≥ y`.

### Lỗi 5: hai con trỏ chỉ nhích một bước khi gặp nhóm trùng

In [26]:
bad = next((a, t) for a, t in tests if bug_two_pointers_step_one(a, t) != oracle(a, t))
print(bad, "->", bug_two_pointers_step_one(*bad), "| đúng:", oracle(*bad))

([3, 2, 3, 4, 3, 4, 2, 4], 9) -> 7 | đúng: 19


**Vì sao sai:** khi `a[j] ≠ a[k]` có thể có **nhiều** chỉ số cùng giá trị ở mỗi phía, mỗi cặp là một bộ khác nhau; khi `a[j] == a[k]` cả đoạn cùng giá trị cho `C(m, 2)` cặp. **Đúng:** gom nhóm `L·R` hoặc `C(m, 2)`.

### Lỗi 6 (bonus): quên modulo

`C(3000, 3) ≈ 4.5·10^9 > 10^9 + 7`; đề yêu cầu modulo (và ở Java/C++ phép nhân `a·b·c` có thể tràn `int`: dùng `long`).

In [27]:
print("không mod:", bug_no_mod(big_eq, 150), "| đúng (mod):", Solution().threeSumMulti(big_eq, 150))

không mod: 4495501000 | đúng (mod): 495500972


**Vì sao sai:** kết quả vượt `10^9 + 7`. **Đúng:** `% (10**9 + 7)`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "đếm bộ chỉ số có tổng bằng `target`, giá trị nhỏ nhưng nhiều phần tử trùng" → **đếm theo giá trị + tổ hợp**; hoặc sắp xếp + hai con trỏ gom nhóm trùng; hoặc DP "chọn đúng `k` phần tử".

**Mẹo:**

1. Phân biệt đếm **bộ giá trị** (3Sum) với đếm **bộ chỉ số** (bài này).
2. Ép `x ≤ y ≤ z` để mỗi đa tập chỉ xét một lần.
3. Bốn dạng: khác nhau, `x = y`, `y = z`, cả ba bằng nhau.
4. Tổ hợp: `C(m,2)`, `C(m,3)` bằng số nguyên (chia hết đúng).
5. Giá trị nhỏ (`≤ 100`) ⇒ đếm bằng mảng, `O(101²)`.
6. Modulo ở cuối; dùng `long` ở ngôn ngữ 32-bit.

**Họ bài cùng pattern:** 3Sum (15), Two Sum (1), 4Sum (18), Number of Good Pairs (1512), 3Sum Smaller (259), Count Number of Nice Subarrays, Subarray Sum Equals K (560), Count Triplets That Can Form Two Arrays of Equal XOR (1442).

## 15. Biến thể

1. **Số bộ giá trị khác nhau** (3Sum, 15): đếm các đa tập `{x, y, z}` có tổng `target`.
2. **`k`-Sum bội số** cho `k` bất kỳ (DP chọn đúng `k` phần tử).
3. **Số cặp** (2Sum bội số): `x ≤ y`, `y = target − x`.
4. **Tổng nhỏ hơn `target`** (259, Premium): đếm bộ chỉ số có tổng `< target`.
5. **Đa tập `k` phần tử** với mảng có phần tử âm (dịch giá trị).

Cài biến thể 1, 2, 3 và đối chiếu vét cạn:

In [28]:
def distinct_triplets(arr, target):
    vals = sorted(set(arr))
    cnt = Counter(arr)
    res = 0
    for x in vals:
        for y in vals:
            if y < x:
                continue
            z = target - x - y
            if z < y or z not in cnt:
                continue
            need = Counter([x, y, z])
            if all(cnt[v] >= m for v, m in need.items()):
                res += 1
    return res


def distinct_triplets_brute(arr, target):
    return len({tuple(sorted(c)) for c in itertools.combinations(arr, 3) if sum(c) == target})


def ksum_multiplicity(arr, k, target):
    """Biến thể 2: số tập k chỉ số có tổng target (DP chọn đúng c phần tử); modulo."""
    ways = [[0] * (target + 1) for _ in range(k + 1)]
    ways[0][0] = 1
    for x in arr:
        for c in range(k, 0, -1):
            for s in range(target, x - 1, -1):
                ways[c][s] += ways[c - 1][s - x]
    return ways[k][target] % MOD


def two_sum_multiplicity(arr, target):
    cnt = Counter(arr)
    res = 0
    for x in cnt:
        y = target - x
        if y < x or y not in cnt:
            continue
        res += math.comb(cnt[x], 2) if x == y else cnt[x] * cnt[y]
    return res % MOD


rng = random.Random(15)
for _ in range(400):
    a = [rng.randint(0, 6) for _ in range(rng.randint(3, 10))]
    t = rng.randint(0, 18)
    assert distinct_triplets(a, t) == distinct_triplets_brute(a, t)
    for k in (1, 2, 3, 4):
        assert ksum_multiplicity(a, k, t) == sum(1 for c in itertools.combinations(a, k) if sum(c) == t) % MOD
    assert two_sum_multiplicity(a, t) == ksum_multiplicity(a, 2, t)
assert ksum_multiplicity([1, 1, 2, 2, 3, 3, 4, 4, 5, 5], 3, 8) == 20
print("3Sum bộ giá trị khác nhau, k-Sum bội số (k = 1..4), 2Sum bội số: khớp vét cạn trên 400 mảng")

3Sum bộ giá trị khác nhau, k-Sum bội số (k = 1..4), 2Sum bội số: khớp vét cạn trên 400 mảng


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi bộ chỉ số | ba vòng lặp | `O(n³)` | `O(1)` | oracle |
| 2. Sắp xếp + hai con trỏ | gom nhóm trùng `L·R`, `C(m,2)` | `O(n²)` | `O(1)` | không giả định miền giá trị nhỏ |
| 3. Đếm theo giá trị | `x ≤ y ≤ z`, bốn dạng tổ hợp | `O(n + 101²)` | `O(101)` | **đáp án chính** |
| 4. DP chọn 3 phần tử | `ways[c][s]` | `O(3·n·target)` | `O(3·target)` | tổng quát `k-Sum` |

Cả bốn khớp oracle `combinations` trên 59 840 bộ nhỏ và 1 500 bộ ngẫu nhiên, và khớp nhau ở `n = 3000` (mục 12).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [29]:
print(Solution().threeSumMulti([1, 1, 1, 2, 2], 4), Solution().threeSumMulti([0, 0, 0, 0], 0), Solution().threeSumMulti([1, 2, 3, 4], 100), distinct_triplets([1, 1, 2, 2, 3], 6))

6 4 0 1


### Bài 2 (tính tay)

`arr = [1, 1, 2, 3, 3, 3]`, `target = 7`. (a) Liệt kê các bộ giá trị `x ≤ y ≤ z` có tổng `7`. (b) Số bộ chỉ số của từng bộ? (c) Tổng? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện đếm theo giá trị:

In [30]:
def counting_todo(arr, target):
    cnt = [0] * 101
    for v in arr:
        cnt[v] += 1
    ans = 0
    for x in range(101):
        for y in range(x, 101):
            z = target - x - y
            if z < y or z > 100:
                continue
            pass        # TODO: cộng số bộ chỉ số theo 4 dạng: x==y==z, x==y, y==z, khác nhau
    return ans % MOD


# Khi làm xong, bỏ comment để kiểm tra:
# assert [counting_todo(list(a), t) for (a, t), _ in examples] == [20, 12, 1]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **DP chọn 3 phần tử**:

In [31]:
class MySolution:
    def threeSumMulti(self, arr, target):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().threeSumMulti(list(a), t) == w for (a, t), w in examples)

### Bài 5 (mini challenge): 4Sum II (454)

Cho bốn mảng `A, B, C, D` cùng độ dài `n`, đếm số bộ `(i, j, k, l)` với `A[i] + B[j] + C[k] + D[l] = 0`. Viết `four_sum_count(A, B, C, D)` trong `O(n²)` bằng bảng băm tổng hai mảng đầu, và kiểm tra bằng vét cạn.

In [32]:
def four_sum_count(A, B, C, D):
    pass            # TODO


# Khi làm xong:
# assert four_sum_count([1, 2], [-2, -1], [-1, 2], [0, 2]) == 2

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: đếm theo giá trị và tổ hợp.
- Mức 2: `[1,1,1,2,2]`, `4`: `(1,1,2)` ⇒ `C(3,2)·2 = 6`; `[0,0,0,0]`, `0` ⇒ `C(4,3) = 4`.
- Mức 3: kết quả `6 4 0 1`: không có bộ tổng `100`; 3Sum bộ giá trị khác nhau của `[1,1,2,2,3]` tổng `6`: chỉ `{1,2,3}` ⇒ `1`.

**Bài 2**

- Mức 1: `cnt = {1:2, 2:1, 3:3}`.
- Mức 2: các bộ `x ≤ y ≤ z` tổng `7`: `(1,3,3)`, `(2,2,3)`? (`2` chỉ có một phần tử nên `C(1,2) = 0`).
- Mức 3: `(1,3,3)`: `2 · C(3,2) = 6`; còn `(2,2,3)` cho `0`; `(1,... )` khác: không có. Tổng `6`.

**Bài 3**

- Mức 1: `a, b, c = cnt[x], cnt[y], cnt[z]`.
- Mức 2: `x == y == z` → `a*(a-1)*(a-2)//6`; `x == y` → `a*(a-1)//2*c`; `y == z` → `a*(b*(b-1)//2)`.
- Mức 3: ngược lại `a*b*c`.

**Bài 4**

- Mức 1: `ways[c][s]` với `c = 0..3`.
- Mức 2: duyệt `c` giảm dần, `s` giảm dần.
- Mức 3: trả `ways[3][target] % MOD`.

**Bài 5**

- Mức 1: băm `Counter(a + b for a in A for b in B)`.
- Mức 2: với mỗi `c + d`, cộng `table[-(c + d)]`.
- Mức 3: `O(n²)` thời gian và bộ nhớ.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [33]:
vals = (Solution().threeSumMulti([1, 1, 1, 2, 2], 4), Solution().threeSumMulti([0, 0, 0, 0], 0), Solution().threeSumMulti([1, 2, 3, 4], 100), distinct_triplets([1, 1, 2, 2, 3], 6))
assert vals == (6, 4, 0, 1) == (oracle([1, 1, 1, 2, 2], 4), oracle([0, 0, 0, 0], 0), oracle([1, 2, 3, 4], 100), distinct_triplets_brute([1, 1, 2, 2, 3], 6))
print("Bài 1 OK:", vals)

Bài 1 OK: (6, 4, 0, 1)


### Bài 2

(a) `(1,3,3)` và `(2,2,3)`. (b) `(1,3,3)`: `2 · C(3,2) = 6`; `(2,2,3)`: `C(1,2) · 3 = 0`. (c) Tổng `6`.

In [34]:
a, t = [1, 1, 2, 3, 3, 3], 7
cnt = Counter(a)
assert cnt[1] * math.comb(cnt[3], 2) == 6 and math.comb(cnt[2], 2) * cnt[3] == 0
assert Solution().threeSumMulti(a, t) == oracle(a, t) == 6
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [35]:
def counting_answer(arr, target):
    cnt = [0] * 101
    for v in arr:
        cnt[v] += 1
    ans = 0
    for x in range(101):
        for y in range(x, 101):
            z = target - x - y
            if z < y or z > 100:
                continue
            a, b, c = cnt[x], cnt[y], cnt[z]
            if x == y == z:
                ans += a * (a - 1) * (a - 2) // 6
            elif x == y:
                ans += a * (a - 1) // 2 * c
            elif y == z:
                ans += a * (b * (b - 1) // 2)
            else:
                ans += a * b * c
    return ans % MOD


assert [counting_answer(list(a), t) for (a, t), _ in examples] == [20, 12, 1]
for a, t in tests[:300]:
    assert counting_answer(a, t) == oracle(a, t)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [36]:
class MySolutionAnswer:
    def threeSumMulti(self, arr, target):
        ways = [[0] * (target + 1) for _ in range(4)]
        ways[0][0] = 1
        for x in arr:
            for c in range(3, 0, -1):
                for s in range(target, x - 1, -1):
                    ways[c][s] += ways[c - 1][s - x]
        return ways[3][target] % (10 ** 9 + 7)


assert all(MySolutionAnswer().threeSumMulti(list(a), t) == w for (a, t), w in examples)
for a, t in tests[:300]:
    assert MySolutionAnswer().threeSumMulti(a, t) == oracle(a, t)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [37]:
def four_sum_count_answer(A, B, C, D):
    table = Counter(a + b for a in A for b in B)
    return sum(table[-(c + d)] for c in C for d in D)


def four_sum_brute(A, B, C, D):
    return sum(1 for a in A for b in B for c in C for d in D if a + b + c + d == 0)


assert four_sum_count_answer([1, 2], [-2, -1], [-1, 2], [0, 2]) == 2
rng = random.Random(5)
for _ in range(300):
    n = rng.randint(1, 6)
    A, B, C, D = ([rng.randint(-3, 3) for _ in range(n)] for _ in range(4))
    assert four_sum_count_answer(A, B, C, D) == four_sum_brute(A, B, C, D)
print("Bài 5 OK: khớp vét cạn")

Bài 5 OK: khớp vét cạn


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (đếm bộ chỉ số tổng `target`, giá trị nhỏ → đếm theo giá trị + tổ hợp):**

```python
class Solution:
    def threeSumMulti(self, arr: List[int], target: int) -> int:
        cnt = [0] * 101
        for v in arr:
            cnt[v] += 1
        ans = 0
        for x in range(101):
            for y in range(x, 101):                  # ép x <= y <= z
                z = target - x - y
                if z < y or z > 100:
                    continue
                a, b, c = cnt[x], cnt[y], cnt[z]
                if x == y == z:   ans += a * (a - 1) * (a - 2) // 6
                elif x == y:      ans += a * (a - 1) // 2 * c
                elif y == z:      ans += a * (b * (b - 1) // 2)
                else:             ans += a * b * c
        return ans % (10 ** 9 + 7)
```

**Checklist trước khi submit:**

- [ ] Đếm bộ **chỉ số** (nhiều bộ cùng bộ giá trị tính riêng) bằng tổ hợp?
- [ ] Ép `x ≤ y ≤ z` (không đếm lặp hoán vị)?
- [ ] Bốn dạng: khác nhau, `x = y`, `y = z`, cả ba bằng nhau; `C(m,2) = m(m−1)/2`, `C(m,3) = m(m−1)(m−2)/6`?
- [ ] `z` trong `[y, 100]`?
- [ ] `% (10**9 + 7)` (và `long` ở ngôn ngữ 32-bit)?
- [ ] Đã thử 3 ví dụ, `n = 3`, toàn bằng nhau (`C(n,3)`), không có bộ nào?
- [ ] Đúng tên `class Solution`, `threeSumMulti(self, arr, target)`, không in debug.